<a href="https://colab.research.google.com/github/brunopn-code/workflow-performance-analytics/blob/main/notebooks/03_sql_kpi_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL KPI Analysis

This notebook creates business-oriented process KPIs from the BPI Challenge 2012 event log.

The goal is to translate the exploratory analysis into reusable SQL metrics that could support a workflow performance dashboard.

The analysis focuses on:

- case duration
- delayed case rate
- activity frequency
- repeated activities
- transition frequency
- waiting time between activities
- resource workload

In [2]:
!pip install duckdb

In [3]:
import pandas as pd
import duckdb

In [4]:
df = pd.read_csv("bpi_2012_events.csv")

In [5]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="mixed",
    utc=True
)

df["case_registration_date"] = pd.to_datetime(
    df["case_registration_date"],
    format="mixed",
    utc=True
)

In [6]:
con = duckdb.connect()
con.register("events", df)

## Dataset Overview KPI

This query summarizes the size of the event log, including total events, total cases, unique activities, and unique resources.

In [7]:
con.sql("""
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT case_id) AS total_cases,
    COUNT(DISTINCT activity) AS total_activities,
    COUNT(DISTINCT resource) AS total_resources,
    MIN(timestamp) AS first_event_timestamp,
    MAX(timestamp) AS last_event_timestamp
FROM events
""").df()

,total_events,total_cases,total_activities,total_resources,first_event_timestamp,last_event_timestamp
0,262200,13087,24,68,2011-10-01 00:38:44.546000+00:00,2012-03-14 16:04:54.681000+00:00


## Case Duration KPI

This query calculates the duration of each loan application case from its first event to its last event.

In [13]:
case_duration_sql = con.sql("""
WITH case_times AS (
    SELECT
        case_id,
        MIN(timestamp) AS case_start,
        MAX(timestamp) AS case_end
    FROM events
    GROUP BY case_id
)

SELECT
    case_id,
    case_start,
    case_end,
    DATE_DIFF('hour', case_start, case_end) / 24.0 AS duration_days
FROM case_times
ORDER BY duration_days DESC
""").df()

case_duration_sql["duration_days"] = case_duration_sql["duration_days"].astype(int)
case_duration_sql.head()

,case_id,case_start,case_end,duration_days
0,173694,2011-10-01 08:10:30.287000+00:00,2012-02-15 12:29:26.299000+00:00,137
1,179591,2011-10-24 23:19:38.689000+00:00,2012-01-24 09:15:14.850000+00:00,91
2,188485,2011-11-23 15:56:48.528000+00:00,2012-02-19 09:15:24.248000+00:00,87
3,189805,2011-11-29 12:31:07.629000+00:00,2012-02-23 09:33:31.826000+00:00,85
4,183405,2011-11-09 11:54:58.210000+00:00,2012-02-01 18:36:24.466000+00:00,84


In [14]:
case_duration_sql["duration_days"].describe()

,duration_days
count,13087.000000
mean,8.302972
std,11.958221
min,0.000000
25%,0.000000
50%,0.000000
75%,14.000000
max,137.000000


In [15]:
con.register("case_duration", case_duration_sql)

In [16]:
case_duration_category_sql = con.sql("""
SELECT
    case_id,
    duration_days,
    CASE
        WHEN duration_days = 0 THEN 'Same day'
        WHEN duration_days <= 1 THEN '1 day'
        WHEN duration_days <= 14 THEN '2-14 days'
        WHEN duration_days <= 40 THEN '15-40 days'
        ELSE 'Over 40 days'
    END AS duration_category
FROM case_duration
""").df()

case_duration_category_sql.head()

,case_id,duration_days,duration_category
0,173694,137,Over 40 days
1,179591,91,Over 40 days
2,188485,87,Over 40 days
3,189805,85,Over 40 days
4,183405,84,Over 40 days


In [17]:
con.register("case_duration_category", case_duration_category_sql)

con.sql("""
SELECT
    duration_category,
    COUNT(*) AS total_cases,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS case_percentage
FROM case_duration_category
GROUP BY duration_category
ORDER BY total_cases DESC
""").df()

,duration_category,total_cases,case_percentage
0,Same day,6762,51.67
1,15-40 days,2884,22.04
2,2-14 days,2833,21.65
3,1 day,408,3.12
4,Over 40 days,200,1.53
